# <a id='toc1_'></a>[Using gene symbol categories to resolve ambiguous gene symbols](#toc0_)

Gene symbol categories are determined by the relationship the gene symbol has with its associated gene concept. These categories are ranked to show priority and used to assign ambiguous gene symbols to unique gene concepts.

This notebook does two things:
1. Collects a set of ambiguous gene symbols that have previously been annotated to unique gene concepts
2. Validates a ranked list of gene symbol categories against the set of ambiguous gene symbols

**Table of contents**<a id='toc0_'></a>    
- [Using gene symbol categories to resolve ambiguous gene symbols](#toc1_)    
  - [Collect a set of ambiguous gene symbols that have been annotated to unique gene concepts](#toc1_1_)    
    - [CIViC](#toc1_1_1_)    
      - [Build lookup by CIViC molecular profile ID](#toc1_1_1_1_)    
      - [Make an ambiguous symbol dataframe using alias-alias and alias-primary collisions](#toc1_1_1_2_)    
      - [Remove genes from civic_df that are not involved in collisions (don't have associated ambiguous gene symbols)](#toc1_1_1_3_)    
      - [Query articles for ambiguous symbol using Pubtator3](#toc1_1_1_4_)    
      - [Only 23 documents and 5 gene-symbol pairs, need more](#toc1_1_1_5_)    
    - [DGIdb](#toc1_1_2_)    
    - [James provided raw data of all gene claims](#toc1_1_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import sys
from pathlib import Path

analysis_dir = Path.cwd().parent

if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import importlib
import re

import civicpy.civic as civic
import gene_ids_in_lit.functions as giilfn
import collision_analysis_shared_variables as casv
import pandas as pd
import functions as fn

importlib.reload(giilfn)
importlib.reload(fn)

<module 'functions' from '/Users/rsaxs014/Desktop/gene-harmony-analysis/analysis/ranked_category_resolver/functions.py'>

## <a id='toc1_1_'></a>[Collect a set of ambiguous gene symbols that have been annotated to unique gene concepts](#toc0_)

### <a id='toc1_1_2_'></a>[Using DGIdb gene claims](#toc0_)

James (DGIdb developer) provided raw data of all gene claims, the website download of genes.tsv does not include the "aliases" column that includes supporting information provided by the source

In [ ]:
dgidb_genes_df = pd.read_csv("../../input/gene_claims20260814.csv")

In [19]:
len(dgidb_genes_df)

79767

In [20]:
# Filter the dgidb gene claims so that only those using alias gene symbols remain

alias_dgidb_genes_df = dgidb_genes_df[
    (~dgidb_genes_df["name"].isin(casv.primary_symbol_set))
    & (dgidb_genes_df["name"].isin(casv.alias_symbol_set))
].copy()

In [21]:
len(alias_dgidb_genes_df)

752

In [22]:
# Filter the dgidb gene claims so that only those using ambiguous gene symbols remain

ambiguous_alias_dgidb_genes_df = alias_dgidb_genes_df[
    alias_dgidb_genes_df["name"].isin(
        collision_df["ambiguous_symbol"].explode()
    )
].copy()

In [23]:
ambiguous_alias_dgidb_genes_df

,name,nomenclature,source_db_name,normalized_gene_id,aliases
153,DAC,Gene Name,DrugBank,hgnc:17,DAC_ACTSP|UNIPROT:P39045|X64790
599,ARSC,Gene Name,DrugBank,hgnc:716,ARSC_STAAU|ARSC1_ECOLX|GENBANK:150729|GENBANK:...
910,AMY,Gene Name,DrugBank,hgnc:17187,AMY_PSEHA|GENBANK:2879820|UNIPROT:P29957|X58627
3115,CCRL1,Gene Symbol,dGene,hgnc:1611,CC-CKR-11|CCBP2|CCR-11|CCR10|CCR11|CCX CKR|CCX...
4020,HCA1,Gene Symbol,NCBI,ncbigene:266790,"AH|HCA|Hypercalciuria, absorptive, 1|ncbigene:..."
...,...,...,...,...,...
76152,ACS3,Gene Name,DrugBank,hgnc:12428,Q8I0X2_PLAF7|UNIPROT:Q8I0X2
77691,ENV,Gene Symbol,ChEMBL,hgnc:39031,CHEMBL:CHEMBL1293311|CHEMBL:CHEMBL2362987|CHEM...
78558,USP17,Gene Symbol,dGene,hgnc:12615,NCBIGENE:391627|RS447|USP17A|USP17H|USP17I|USP...
78877,HCP,Gene Name,DrugBank,hgnc:2321,GENBANK:49286|HCP_DESDA|HCP_DESVH|UNIPROT:P311...


In [24]:
# Don't need this column
ambiguous_alias_dgidb_genes_df = ambiguous_alias_dgidb_genes_df.drop(columns=["nomenclature"])

In [25]:
# # Split pipe-delimited alias/identifier (aliases column) values into individual entries

# items = (
#     ambiguous_alias_dgidb_genes_df["aliases"]
#     .dropna()
#     .str.split("|")
#     .explode()
#     .str.strip()
# )

In [26]:
# # Keep entries containing ":" and extract everything before the first ":"
# # These indicate an identifier with a source prefix

# prefixes = (
#     items[items.str.contains(":", regex=False)]
#     .str.split(":", n=1)
#     .str[0]
# )

In [27]:
# # Visual summary to investigate the prefixes used

# unprefixed = items[~items.str.contains(":", regex=False)]
# prefixed = items[items.str.contains(":", regex=False)].to_frame("value")

# prefixed["prefix"] = prefixed["value"].str.split(":", n=1).str[0]

# summary = (
#     prefixed.groupby("prefix")
#     .agg(
#         count=("value", "size"),
#         example=("value", "first")
#     )
#     .sort_values("count", ascending=False)
# )

# print(summary)

In [28]:
# # Chose to use only identifiers from NCBI Gene, HGNC, and Ensembl

# prefixes = ["hgnc", "ensembl", "ncbigene", "ncbi.gene"]

# pattern = r"(?:^|\|)(?:" + "|".join(map(re.escape, prefixes)) + r"):"

# mask = ambiguous_alias_dgidb_genes_df["aliases"].str.contains(
#     pattern,
#     case=False,
#     regex=True,
#     na=False
# )

# ambiguous_alias_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_df[mask]

In [29]:
# # Removing the source prefix for easier comparison

# ambiguous_alias_dgidb_genes_with_identifiers_df["HGNC_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["aliases"].str.extract(
#     r"(?:^|\|)(?:hgnc|HGNC):([^|]+)",
#     expand=False
# )

# ambiguous_alias_dgidb_genes_with_identifiers_df["NCBI_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["aliases"].str.extract(
#     r"(?:^|\|)(?:ncbigene|NCBIGENE|NCBI\.GENE):([^|]+)",
#     expand=False
# )

# ambiguous_alias_dgidb_genes_with_identifiers_df["ENSG_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["aliases"].str.extract(
#     r"(?:^|\|)(?:ensembl|ENSEMBL):(ENSG[^|]+)",
#     expand=False
# )

In [30]:
ambiguous_alias_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_df[ambiguous_alias_dgidb_genes_df["aliases"].notna()]

ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_HGNC_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_gene_id"].str.extract(
    r"(?i)hgnc:([^|]+)",
    expand=False,
)

ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_NCBI_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_gene_id"].str.extract(
    r"(?i)ncbigene:([^|]+)",
    expand=False,
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_33262/2408291289.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_HGNC_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_gene_id"].str.extract(
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_33262/2408291289.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_NCBI_ID"] = ambiguous_alias_dgidb_genes_w

In [31]:
ambiguous_alias_dgidb_genes_with_identifiers_df

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID
153,DAC,DrugBank,hgnc:17,DAC_ACTSP|UNIPROT:P39045|X64790,17,NaN
599,ARSC,DrugBank,hgnc:716,ARSC_STAAU|ARSC1_ECOLX|GENBANK:150729|GENBANK:...,716,NaN
910,AMY,DrugBank,hgnc:17187,AMY_PSEHA|GENBANK:2879820|UNIPROT:P29957|X58627,17187,NaN
3115,CCRL1,dGene,hgnc:1611,CC-CKR-11|CCBP2|CCR-11|CCR10|CCR11|CCX CKR|CCX...,1611,NaN
4020,HCA1,NCBI,ncbigene:266790,"AH|HCA|Hypercalciuria, absorptive, 1|ncbigene:...",NaN,266790
...,...,...,...,...,...,...
76152,ACS3,DrugBank,hgnc:12428,Q8I0X2_PLAF7|UNIPROT:Q8I0X2,12428,NaN
77691,ENV,ChEMBL,hgnc:39031,CHEMBL:CHEMBL1293311|CHEMBL:CHEMBL2362987|CHEM...,39031,NaN
78558,USP17,dGene,hgnc:12615,NCBIGENE:391627|RS447|USP17A|USP17H|USP17I|USP...,12615,NaN
78877,HCP,DrugBank,hgnc:2321,GENBANK:49286|HCP_DESDA|HCP_DESVH|UNIPROT:P311...,2321,NaN


#### This yields 102 gene claims that normalize ambiguous gene symbol to gene concepts

### Validate ranked gene symbol categories

#### This file contains gene-to-symbol pairs annotated with relationship categories

In [32]:
capture_df = pd.read_csv("../../output/summary_df.csv")

In [33]:
# Currently the values in the ID columns are just strings that look like sets, need to convert to actual sets

for col in ["HGNC_ID", "NCBI_ID", "ENSG_ID"]:
    capture_df[col] = capture_df[col].apply(fn.to_set)

#### Rank the relationship categories

In [34]:
rank_order = ['Primary Gene Symbol',
                'Previous Symbol',
                'Clone Name Symbol',
                "Gene Identifier Symbol",
                "Placeholder Symbol",
                "Ortholog Symbol",
                "Alternate Abbreviation Symbol",
                "Withdrawn Ortholog Symbol",
                "Prefix Condition Symbol",
                "Gene Group Symbol",
                "Protein Mass Symbol",
                "Related Gene Symbol",
                "Gene Neighbor Symbol",
                "Gene Interaction Symbol"]

rank_map = {category: i for i, category in enumerate(rank_order)}

#### Use the ranked categories to resolve the ambiguous gene symbol to a gene concept

In [35]:
# Run the function on every ambiguous DGIdb row
result_cols = [
    "rank_match",
    "rank_status",
    "winning_category",
    "winning_HGNC_ID"
]

ambiguous_alias_dgidb_genes_with_identifiers_df[result_cols] = (
    ambiguous_alias_dgidb_genes_with_identifiers_df.apply(
        fn.check_rank_match,
        axis=1,
        capture_df=capture_df,
        rank_map=rank_map,
    )
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_33262/334899157.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df[result_cols] = (
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_33262/334899157.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df[result_cols] = (
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_33262/334899157.py:9: SettingWithCopyWarning: 
A value is trying to b

In [36]:
# Covert the rank_match type to boolean

ambiguous_alias_dgidb_genes_with_identifiers_df["rank_match"] = (
    ambiguous_alias_dgidb_genes_with_identifiers_df["rank_match"]
    .astype("boolean")
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_33262/1773379390.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df["rank_match"] = (


In [37]:
ambiguous_alias_dgidb_genes_with_identifiers_df

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID,rank_match,rank_status,winning_category,winning_HGNC_ID
153,DAC,DrugBank,hgnc:17,DAC_ACTSP|UNIPROT:P39045|X64790,17,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:17}
599,ARSC,DrugBank,hgnc:716,ARSC_STAAU|ARSC1_ECOLX|GENBANK:150729|GENBANK:...,716,NaN,True,matched,Previous Symbol,{HGNC:716}
910,AMY,DrugBank,hgnc:17187,AMY_PSEHA|GENBANK:2879820|UNIPROT:P29957|X58627,17187,NaN,<NA>,no captured as,<NA>,<NA>
3115,CCRL1,dGene,hgnc:1611,CC-CKR-11|CCBP2|CCR-11|CCR10|CCR11|CCX CKR|CCX...,1611,NaN,True,matched,Previous Symbol,{HGNC:1611}
4020,HCA1,NCBI,ncbigene:266790,"AH|HCA|Hypercalciuria, absorptive, 1|ncbigene:...",NaN,266790,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:4532}
...,...,...,...,...,...,...,...,...,...,...
76152,ACS3,DrugBank,hgnc:12428,Q8I0X2_PLAF7|UNIPROT:Q8I0X2,12428,NaN,True,matched,Previous Symbol,{HGNC:12428}
77691,ENV,ChEMBL,hgnc:39031,CHEMBL:CHEMBL1293311|CHEMBL:CHEMBL2362987|CHEM...,39031,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:53424}
78558,USP17,dGene,hgnc:12615,NCBIGENE:391627|RS447|USP17A|USP17H|USP17I|USP...,12615,NaN,True,matched,Previous Symbol,{HGNC:12615}
78877,HCP,DrugBank,hgnc:2321,GENBANK:49286|HCP_DESDA|HCP_DESVH|UNIPROT:P311...,2321,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:2321}


In [51]:
ambiguous_alias_dgidb_genes_with_identifiers_df["winning_category"].value_counts()

winning_category
Previous Symbol                  45
Alternate Abbreviation Symbol    21
Ortholog Symbol                   6
Gene Group Symbol                 5
Withdrawn Ortholog Symbol         3
Prefix Condition Symbol           1
Name: count, dtype: int64

In [39]:
capture_df[capture_df["gene_symbol"] == "DAC"]

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
73,73,{HGNC:17},{ENSG00000114771},{GENE ID:13},AADAC,DAC,T,Alternate Abbreviation Symbol
34037,34037,{HGNC:10847},{ENSG00000107829},{GENE ID:6468},FBXW4,DAC,T,Withdrawn Ortholog Symbol


In [40]:
matched_ambiguous_alias_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_with_identifiers_df[ambiguous_alias_dgidb_genes_with_identifiers_df["rank_status"] == "matched"]
matched_ambiguous_alias_dgidb_genes_with_identifiers_df

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID,rank_match,rank_status,winning_category,winning_HGNC_ID
153,DAC,DrugBank,hgnc:17,DAC_ACTSP|UNIPROT:P39045|X64790,17,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:17}
599,ARSC,DrugBank,hgnc:716,ARSC_STAAU|ARSC1_ECOLX|GENBANK:150729|GENBANK:...,716,NaN,True,matched,Previous Symbol,{HGNC:716}
3115,CCRL1,dGene,hgnc:1611,CC-CKR-11|CCBP2|CCR-11|CCR10|CCR11|CCX CKR|CCX...,1611,NaN,True,matched,Previous Symbol,{HGNC:1611}
5668,APR,DrugBank,hgnc:6692,GENBANK:142526|GENBANK:5921206|K02496|SUBD_BAC...,6692,NaN,True,matched,Previous Symbol,{HGNC:6692}
5868,DUSP13,dGene,hgnc:19681,BEDP|DUSP13A|DUSP13B|MDSP|NCBIGENE:51207|SKRP4...,19681,NaN,True,matched,Previous Symbol,{HGNC:19681}
7010,CMK,DrugBank,hgnc:7098,GENBANK:42839|KCY_ECOLI|UNIPROT:P0A6I0|X00785,7098,NaN,True,matched,Previous Symbol,{HGNC:7098}
9018,NOS,DrugBank,hgnc:7872,BA000033|D86417|GENBANK:21205025|GENBANK:24432...,7872,NaN,True,matched,Previous Symbol,{HGNC:7872}
9590,MARS,Pharos,hgnc:6898,"METHIONINE--TRNA LIGASE, CYTOPLASMIC|UNIPROT:P...",6898,NaN,True,matched,Previous Symbol,{HGNC:6898}
10096,L5,DrugBank,hgnc:10360,Q64822_9ADEN|Q64823_9ADEN|UNIPROT:Q64822|UNIPR...,10360,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:10360}
15775,PTA,DrugBank,hgnc:21290,GENBANK:580883|PTAS_BACSU|UNIPROT:P39646|X73124,21290,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:21290}


In [41]:
matched_ambiguous_alias_dgidb_genes_with_identifiers_df["winning_category"].value_counts()

winning_category
Previous Symbol                  44
Alternate Abbreviation Symbol     8
Ortholog Symbol                   4
Gene Group Symbol                 2
Prefix Condition Symbol           1
Withdrawn Ortholog Symbol         1
Name: count, dtype: int64

#### Investigate the gene claims where the ranked categories mapped the ambiguous gene symbol to a different gene concept than the source

In [42]:
mismatched_ambiguous_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_with_identifiers_df[ambiguous_alias_dgidb_genes_with_identifiers_df["rank_status"] == "ID mismatch"]
mismatched_ambiguous_dgidb_genes_with_identifiers_df

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID,rank_match,rank_status,winning_category,winning_HGNC_ID
4020,HCA1,NCBI,ncbigene:266790,"AH|HCA|Hypercalciuria, absorptive, 1|ncbigene:...",NaN,266790,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:4532}
4337,ENV,DrugBank,hgnc:39031,ENV_HV1BN|ENV_HV1Y2|ENV_SIVMK|M21098|UNIPROT:P...,39031,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:53424}
4550,ACT,NCBI,ncbigene:389036,ncbigene:389036,NaN,389036,False,ID mismatch,Gene Group Symbol,{HGNC:17780}
5936,MST,DrugBank,hgnc:29678,AJ313201|Q7K9G0_LEIMA|UNIPROT:Q7K9G0,29678,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:7223}
9728,GL,ChEMBL,hgnc:14942,CHEMBL:CHEMBL2364696|ENVELOPE GLYCOPROTEIN L|U...,14942,NaN,False,ID mismatch,Ortholog Symbol,{HGNC:21652}
13166,MOP,DrugBank,hgnc:14449,GENBANK:853817|MOP_DESGI|UNIPROT:Q46509|X77222,14449,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:8156}
13591,ALR,ChEMBL,hgnc:380,ALANINE RACEMASE|CHEMBL:CHEMBL2031|UNIPROT:P9WQA9,380,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:4236}
13980,CAMK,DrugBank,hgnc:1464,AF323755|CAMK_RHOSO|UNIPROT:Q93TU6,1464,NaN,False,ID mismatch,Gene Group Symbol,{HGNC:1463}
16683,PARC,DrugBank,hgnc:10616,GENBANK:147106|GENBANK:1490399|GENBANK:1574370...,10616,NaN,False,ID mismatch,Withdrawn Ortholog Symbol,{HGNC:15982}
16814,TGT,DrugBank,hgnc:12612,GENBANK:498141|L33777|TGT_ZYMMO|UNIPROT:P28720,12612,NaN,False,ID mismatch,Ortholog Symbol,{HGNC:23797}


In [43]:
capture_df[capture_df["gene_symbol"]=="LPD"]

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
1105,1105,{HGNC:29567},{ENSG00000103740},{GENE ID:23205},ACSBG1,LPD,T,Withdrawn Ortholog Symbol
97006,97006,{HGNC:14436},{ENSG00000173166},{GENE ID:65059},RAPH1,LPD,F,NaN


In [44]:
capture_df[capture_df["gene_symbol"]=="env"]

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
31253,31253,{HGNC:39025},{},{GENE ID:100775105},ERVK-18,env,F,NaN
31266,31266,{HGNC:39031},{},{GENE ID:100616444},ERVK-20,env,F,NaN
31299,31299,{HGNC:53424},{},{GENE ID:110006328},ERVK-32,env,T,Alternate Abbreviation Symbol


In [45]:
# Filter the annotated gene-to-symbol pairs so that only those with more than one relationship remain

multi_category_genes = (
    capture_df
    .dropna(subset=["captured as:"])
    .groupby("gene_symbol")["captured as:"]
    .nunique()
)

multi_category_genes = multi_category_genes[
    multi_category_genes > 1
].index

In [46]:
impacted_rows = ambiguous_alias_dgidb_genes_with_identifiers_df[
    ambiguous_alias_dgidb_genes_with_identifiers_df["name"].isin(multi_category_genes)
]

In [47]:
impacted_rows[impacted_rows["rank_status"] == "ID mismatch"]

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID,rank_match,rank_status,winning_category,winning_HGNC_ID
67417,PAR4,dGene,hgnc:29998,NCBIGENE:347745,29998,NaN,False,ID mismatch,Previous Symbol,{HGNC:3540}


In [48]:
capture_df[capture_df["gene_symbol"] == "KRAS"]

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
53689,53689,{HGNC:6407},{ENSG00000133703},{GENE ID:3845},KRAS,KRAS,T,Primary Gene Symbol
81133,81133,{HGNC:7989},{ENSG00000213281},{GENE ID:4893},NRAS,KRAS,T,Ortholog Symbol
